# 05 — A demanding case: MiniBooNE

*Chapter 10 · LightGBM · Notebook 5 of 5 — the capstone*

This is where the chapter pays off. We take a real, larger tabular problem — **MiniBooNE**, a particle-
physics dataset where a detector must tell a neutrino **signal** from **background** — and run an honest,
end-to-end workflow with everything Chapter 10 built: leaf-wise growth (NB 1), GOSS (NB 2), the optimal
categorical split (NB 3, named here — MiniBooNE is all numeric), and the estimator, its knobs, and honest
tuning (NB 4).

The question the whole chapter has been circling lands here: **on a problem where speed can matter, is
LightGBM "better" than XGBoost or scikit-learn's HistGradientBoosting?** The honest answer, measured: the
three histogram boosters are **statistically tied on accuracy**, and the **speed winner is decided by the
matching convention — the shape of the trees you ask for — not by the library's name.** No universal best;
measure on your data.

**Prerequisites.** Chapter 00 (imbalance, PR-AUC, thresholds); this chapter's NB 1–4; Chapters 06/08/09
(random forests, permutation importance).

**What you'll do.** Look at the data; set linear and forest baselines; compare the three boosters at
**matched capacity** and reconcile that against a naive default-vs-default; watch how the speed gap moves
as you dial the row count up; see GOSS's efficiency on real data; ship a tuned, early-stopped LightGBM with
an honestly chosen threshold; and read which features it truly relies on.

## The dataset

MiniBooNE (Roe et al. 2005) gives **50 real-valued detector measurements** per event and asks whether the
event is a **signal** neutrino interaction or **background**. It is **130,064 rows**, **72% background /
28% signal**, large enough that fit time is worth watching. There are no `NaN`s, but the data marks an
**undefined** particle-ID measurement with a **−999 sentinel** — in a small fraction of rows. We will not
impute or drop those: **gradient-boosted trees handle a sentinel value natively** (it becomes its own
split point, exactly the native-missing idea of Chapter 9 NB 3), so the models keep every row.

We will hold out a test set we touch **once**, and carve a validation slice from the training data for
early stopping and for choosing a decision threshold.

In [ ]:
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.datasets import fetch_openml, make_classification
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

# Carried from NB 4: LightGBM prints an [Info] banner per fit, and a capstone fits many
# models. LGBM_VERBOSE = 0 quiets that banner while keeping genuine warnings; set it to 1
# to see every fit's banner. (Never the fully-silent verbose=-1.)
LGBM_VERBOSE = 0

# fetch_openml(as_frame=True) returns a named DataFrame (pandas-first); fitting and
# predicting on the same named columns avoids scikit-learn's spurious feature-name warning.
data = fetch_openml("MiniBooNE", version=1, as_frame=True)
X = data.data
y = (data.target == "True").astype(int)   # 1 = signal (the 28% minority)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
Xtr2, Xval, ytr2, yval = train_test_split(Xtr, ytr, test_size=0.20, stratify=ytr, random_state=SEED)
n_nan = int(X.isna().sum().sum())
n_sentinel = int((X == -999.0).any(axis=1).sum())
print(f"X {X.shape}   train {Xtr.shape}   test {Xte.shape}   (val slice {Xval.shape[0]})")
print(f"signal rate: train {ytr.mean():.3f}   test {yte.mean():.3f}   total NaN: {n_nan}")
print(f"rows with a -999 sentinel: {n_sentinel}  ({n_sentinel / len(X):.2%} of rows)")


In [ ]:
viz.plot_class_balance(y.map({0: "background", 1: "signal"}))
plt.show()


**Read the figure.** Background outnumbers signal roughly 72/28. That imbalance is mild but real, so
plain accuracy is a weak guide (always-predict-background already scores 0.72). We lead with **PR-AUC**
(average precision), the imbalance-aware metric from Chapter 00, and choose the decision threshold
deliberately rather than defaulting to 0.5.

In [ ]:
# A 50-D glimpse: standardize, project to 2 components, scatter a subsample by class.
# The -999 sentinels are extreme values that would dominate a distance-based projection, so we
# set those rows aside FOR THIS PLOT ONLY (the models keep them -- trees handle them natively).
clean = ~(Xtr == -999.0).any(axis=1)
proj = make_pipeline(StandardScaler(), PCA(n_components=2, random_state=SEED))
Z = proj.fit_transform(Xtr[clean])
yc = ytr[clean].to_numpy()
rng = np.random.default_rng(SEED)
idx = rng.choice(len(Z), size=4000, replace=False)
fig, ax = plt.subplots(figsize=(7.0, 5.4))
for cls, name, col in [(0, "background", COLORS["class_b"]), (1, "signal", COLORS["class_a"])]:
    m = yc[idx] == cls
    ax.scatter(Z[idx][m, 0], Z[idx][m, 1], s=8, alpha=0.4, color=col, label=name)
ax.set_xlim(-8, 12)
ax.set_ylim(-7, 12)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("MiniBooNE in 2 principal components (sentinel rows set aside)")
ax.legend(markerscale=2)
plt.show()


**Read the figure.** Projected to two of fifty dimensions (with the −999-sentinel rows set aside),
**signal (purple) gathers on the left** and **background (gold) spreads to the right** — a real shift
along PC 1 — but the two clouds overlap heavily through the middle. A straight linear cut would misclassify
that overlap; there is genuine non-linear structure for a tree ensemble to exploit. (The boosters above
trained on the full data, sentinels included.)

## Baselines first

Before any booster, set the bar: a scaled **logistic regression** (linear) and a **random forest**
(Chapter 06). A model only earns its complexity by beating these.

In [ ]:
def evaluate(name, model, fit_time):
    p = model.predict_proba(Xte)[:, 1]
    ap, roc, acc = (average_precision_score(yte, p), roc_auc_score(yte, p),
                    accuracy_score(yte, p >= 0.5))
    print(f"  {name:30s} fit {fit_time:6.2f}s  PR-AUC {ap:.4f}  ROC-AUC {roc:.4f}  acc {acc:.4f}")
    return {"name": name, "time": fit_time, "ap": ap, "roc": roc, "acc": acc}


def fit_timed(model):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    return model, time.perf_counter() - t0


results = []
m, dt = fit_timed(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))
results.append(evaluate("Logistic (scaled)", m, dt))
m, dt = fit_timed(RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED))
results.append(evaluate("RandomForest (200)", m, dt))


## The three boosters, compared fairly

Leaf-wise growth on histogram bins is not a LightGBM exclusive — it is the **shared modern skeleton**:
scikit-learn's `HistGradientBoosting` grows leaf-wise (`max_leaf_nodes`), XGBoost offers it
(`grow_policy='lossguide'`), and LightGBM makes it the default (`num_leaves`). To compare them *fairly* we
must hold the **tree shape** fixed — otherwise we are comparing different models, not the libraries. So we
fix **one convention**: leaf-wise, **`num_leaves = 31`**, depth unbounded, **300 trees with L2 turned off**
on all three — and read **both fit time and PR-AUC**. (Fit times are single-run wall-clock on one machine:
the **ordering at a fixed tree shape** is the portable lesson, not the absolute seconds.)

In [ ]:
NL, N_EST, LR = 31, 300, 0.1
# A *genuinely* matched convention: leaf-wise, 31 leaves, depth unbounded, exactly 300
# trees, and L2 off on all three (reg_lambda / l2_regularization = 0). HistGBR also gets
# early_stopping=False, so it really builds 300 trees on the full training set (its default
# 'auto' would stop early and carve an internal validation split -- not a matched run).
matched = []
m, dt = fit_timed(LGBMClassifier(n_estimators=N_EST, learning_rate=LR, num_leaves=NL,
                                 max_depth=-1, reg_lambda=0.0, random_state=SEED,
                                 verbose=LGBM_VERBOSE))
matched.append(evaluate("LightGBM (num_leaves=31)", m, dt))
m, dt = fit_timed(XGBClassifier(n_estimators=N_EST, learning_rate=LR, tree_method="hist",
                                grow_policy="lossguide", max_leaves=NL, max_depth=0,
                                reg_lambda=0.0, random_state=SEED))
matched.append(evaluate("XGBoost-hist (lossguide, 31)", m, dt))
m, dt = fit_timed(HistGradientBoostingClassifier(max_iter=N_EST, learning_rate=LR,
                                                 max_leaf_nodes=NL, max_depth=None,
                                                 l2_regularization=0.0, early_stopping=False,
                                                 random_state=SEED))
matched.append(evaluate("HistGBR (max_leaf_nodes=31)", m, dt))


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 5.2))
palette = {"LightGBM (num_leaves=31)": COLORS["model"],
           "XGBoost-hist (lossguide, 31)": COLORS["class_e"],
           "HistGBR (max_leaf_nodes=31)": COLORS["class_c"]}
for r in matched:
    ax.scatter(r["time"], r["ap"], s=140, color=palette[r["name"]], edgecolor=COLORS["text"],
               zorder=3, label=r["name"])
for r in results:   # baselines as reference markers
    ax.scatter(r["time"], r["ap"], s=90, color=COLORS["muted"], marker="s",
               edgecolor=COLORS["text"], zorder=3)
    ax.annotate(r["name"].split()[0], (r["time"], r["ap"]),
                textcoords="offset points", xytext=(6, -10), fontsize=8, color=COLORS["muted"])
ax.set_xlabel("fit time (s)")
ax.set_ylabel("test PR-AUC")
ax.set_title("matched capacity: close on accuracy, LightGBM fastest here")
ax.legend(loc="lower right", fontsize=8)
plt.show()


**Read the figure.** With L2 off and the same 300-tree leaf-wise shape, the three boosters land in a
tight horizontal band — PR-AUC **0.9573–0.9578, a spread of 0.0005** (a tie for any practical purpose) —
and all three sit far above the logistic (0.86) and random forest (0.94) squares. On the time axis they
separate: at this matched capacity **LightGBM is fastest** (≈2.3 s), XGBoost-hist (≈3.0 s) and HistGBR
(≈3.9 s) behind. So *at a fixed tree shape*, LightGBM's implementation is the quickest here, and no booster
is meaningfully more accurate than the others. (Those seconds are one machine's single run — trust the
ordering, not the exact values.)

## A fair warning about "out-of-the-box" comparisons

Run the two libraries at their **defaults** and the story can invert — not because one is better, but
because their **default tree shapes differ**. XGBoost defaults to **depthwise, depth 6** (smaller, shallow
trees); LightGBM defaults to **leaf-wise, num_leaves 31** with unbounded depth (larger trees). Smaller
trees fit faster. So a naive default-vs-default measures tree shape, not the engine.

In [ ]:
m, dt = fit_timed(LGBMClassifier(n_estimators=N_EST, learning_rate=LR,
                                 random_state=SEED, verbose=LGBM_VERBOSE))
_ = evaluate("LightGBM default (leaf-wise 31)", m, dt)
m, dt = fit_timed(XGBClassifier(n_estimators=N_EST, learning_rate=LR, tree_method="hist",
                                max_depth=6, random_state=SEED))
_ = evaluate("XGBoost default (depthwise 6)", m, dt)


**Read the numbers.** At defaults, **XGBoost is now faster** (≈1.0 s vs ≈2.4 s) — its depth-6 trees
are smaller than LightGBM's leaf-wise 31-leaf trees — while PR-AUC stays tied (0.957–0.959). The
lesson is the chapter's honesty bar made concrete: **the speed gap is tree shape, not the library.** At
matched capacity LightGBM wins; at default shapes XGBoost wins; the accuracy is the same either way.

## Does the row count change the winner?

LightGBM was built for **large** data — does its standing improve as `n` grows? We dial a synthetic
problem up and time both conventions: matched capacity (LightGBM vs XGBoost-hist, both num_leaves 31) and
default-vs-default (LightGBM leaf-wise vs XGBoost depthwise-6).

In [ ]:
ns = [50_000, 200_000, 500_000]
lgb_m, xgb_m, lgb_d, xgb_d = [], [], [], []
for n in ns:
    Xs, ys = make_classification(n_samples=n, n_features=50, n_informative=20, n_redundant=10,
                                 flip_y=0.03, random_state=SEED)
    cut = int(0.75 * n)
    Xs_tr, ys_tr = Xs[:cut], ys[:cut]

    def t_fit(model, Xt=Xs_tr, yt=ys_tr):
        t0 = time.perf_counter()
        model.fit(Xt, yt)
        return time.perf_counter() - t0

    lgb_m.append(t_fit(LGBMClassifier(n_estimators=N_EST, learning_rate=LR, num_leaves=NL,
                                      random_state=SEED, verbose=LGBM_VERBOSE)))
    xgb_m.append(t_fit(XGBClassifier(n_estimators=N_EST, learning_rate=LR, tree_method="hist",
                                     grow_policy="lossguide", max_leaves=NL, max_depth=0,
                                     random_state=SEED)))
    lgb_d.append(t_fit(LGBMClassifier(n_estimators=N_EST, learning_rate=LR,
                                      random_state=SEED, verbose=LGBM_VERBOSE)))
    xgb_d.append(t_fit(XGBClassifier(n_estimators=N_EST, learning_rate=LR, tree_method="hist",
                                     max_depth=6, random_state=SEED)))
    print(f"  n={n:7d}: matched LGB {lgb_m[-1]:5.2f}s / XGB {xgb_m[-1]:5.2f}s | "
          f"default LGB {lgb_d[-1]:5.2f}s / XGB {xgb_d[-1]:5.2f}s")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
a1.plot(ns, lgb_m, "o-", color=COLORS["model"], linewidth=2.2, label="LightGBM")
a1.plot(ns, xgb_m, "o-", color=COLORS["class_e"], linewidth=2.2, label="XGBoost-hist")
a1.set_title("matched capacity (num_leaves=31)")
a2.plot(ns, lgb_d, "o-", color=COLORS["model"], linewidth=2.2, label="LightGBM (leaf-wise)")
a2.plot(ns, xgb_d, "o-", color=COLORS["class_e"], linewidth=2.2, label="XGBoost (depth-6)")
a2.set_title("default-vs-default")
for a in (a1, a2):
    a.set_xlabel("training rows (n)")
    a.legend()
a1.set_ylabel("fit time (s)")
plt.show()


**Read the figure.** Two clean stories. **Matched** (left): LightGBM is faster at every `n` we try —
its leaf-wise implementation runs quicker at a fixed tree shape. **Default** (right): XGBoost's
depth-6 trees stay faster across this range, but the gap **narrows** as `n` grows. Pushed offline well
past what a notebook should run — on this same synthetic generator, **beyond the three points plotted
here** — the default ratio falls from ≈2.4× at 50k to **1.1× at 4 million rows**, so LightGBM's leaf-wise
default would overtake only **beyond a few million rows** (Exercise 2 invites you to reproduce this). So
the winner is set by the **convention** (which tree shape you ask for), and only slowly by **scale** —
there is **no universal best**. Pick the convention that fits your accuracy/latency budget and measure it
on your data.

## GOSS on real data

Notebook 2 found that GOSS's edge over a plain uniform subsample needs **concentrated gradients** — on a
small dense toy it was a wash. MiniBooNE is larger and harder; does the edge appear? We compare training on
all rows, with **GOSS** (keep the top 20% by gradient, sample 10% of the rest), and a **uniform 30%**
subsample — all at matched capacity.

In [ ]:
goss = []
for label, kw in [("full (all rows)", {}),
                  ("GOSS (30%)", dict(data_sample_strategy="goss", top_rate=0.2, other_rate=0.1)),
                  ("uniform (30%)", dict(subsample=0.3, subsample_freq=1))]:
    m, dt = fit_timed(LGBMClassifier(n_estimators=N_EST, learning_rate=LR, num_leaves=NL,
                                     random_state=SEED, verbose=LGBM_VERBOSE, **kw))
    goss.append(evaluate(label, m, dt))

fig, ax = plt.subplots(figsize=(6.6, 4.6))
labels = [g["name"] for g in goss]
ax.bar(labels, [g["ap"] for g in goss],
       color=[COLORS["muted"], COLORS["model"], COLORS["class_b"]],
       edgecolor=COLORS["text"], linewidth=0.5)
ax.set_ylim(0.95, 0.96)
ax.set_ylabel("test PR-AUC")
ax.set_title("GOSS matches full data and beats a uniform subsample")
for i, g in enumerate(goss):
    ax.text(i, g["ap"] + 0.0003, f"{g['ap']:.4f}", ha="center")
plt.show()


**Read the figure.** On 30% of the rows (note the **zoomed y-axis** — these gaps are a few
thousandths of PR-AUC), **GOSS matches the full-data PR-AUC** (0.9579 vs 0.9573) and **beats the uniform
subsample** (0.9556) — the edge that was flat on Notebook 2's small dense toy
appears here, exactly as predicted: real, larger data has the imbalanced gradients GOSS exploits. The fit
times are nearly equal (the per-feature histogram build dominates at this width), so on this dataset GOSS
buys *statistical* efficiency more than wall-clock — its time payoff arrives on far larger or wider data.

## A model to ship: tuned LightGBM, early stopping, an honest threshold

Now the deliverable. We grow a slightly wider LightGBM (`num_leaves=63`), let a **validation set stop it**
(no guessing the tree count), choose the decision **threshold on that validation set** (Chapter 00 — never
on the test set), and read the held-out test **once**.

In [ ]:
tuned = LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=63,
                       random_state=SEED, verbose=LGBM_VERBOSE)
tuned.fit(Xtr2, ytr2, eval_set=[(Xval, yval)], eval_metric="average_precision",
          callbacks=[lgb.early_stopping(50)])
print(f"early stopping kept {tuned.best_iteration_} of 2000 trees")

p_test = tuned.predict_proba(Xte)[:, 1]
print(f"held-out: PR-AUC {average_precision_score(yte, p_test):.4f}   "
      f"ROC-AUC {roc_auc_score(yte, p_test):.4f}   acc {accuracy_score(yte, p_test >= 0.5):.4f}")

# choose the F1-optimal threshold on the VALIDATION set, then read the test once
p_val = tuned.predict_proba(Xval)[:, 1]
prec, rec, thr = precision_recall_curve(yval, p_val)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
t_star = float(thr[int(np.argmax(f1s[:-1]))])
print(f"val-chosen F1 threshold {t_star:.3f}: test F1 {f1_score(yte, p_test >= t_star):.4f} "
      f"(threshold 0.5 -> {f1_score(yte, p_test >= 0.5):.4f})")


In [ ]:
yte_arr = yte.to_numpy()
sel = p_test >= t_star
tp = int((sel & (yte_arr == 1)).sum())
op_recall = tp / int((yte_arr == 1).sum())     # recall at the chosen threshold
op_prec = tp / int(sel.sum())                  # precision at the chosen threshold

prec_te, rec_te, _ = precision_recall_curve(yte, p_test)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8))
a1.plot(rec_te, prec_te, color=COLORS["model"], linewidth=2.2)
a1.scatter([op_recall], [op_prec], s=120, color=COLORS["highlight"], edgecolor=COLORS["text"],
           zorder=5, label=f"threshold {t_star:.3f}")
a1.axhline(yte.mean(), color=COLORS["muted"], linestyle=":", linewidth=1,
           label="no-skill (base rate)")
a1.set_xlabel("recall")
a1.set_ylabel("precision")
a1.set_title("test precision-recall curve")
a1.legend(loc="lower left")

cm = confusion_matrix(yte_arr, sel.astype(int))
viz.plot_confusion_matrix(cm, class_names=["background", "signal"], ax=a2)
a2.set_title(f"confusion at threshold {t_star:.3f}")
plt.show()


**Read the figure.** The precision-recall curve hugs the top-right far above the no-skill base rate
(0.28): PR-AUC ≈ 0.958. The validation-chosen threshold (≈0.46, the highlighted point) trades a little
precision for recall versus the default 0.5, lifting test F1 to ≈0.905. The confusion matrix shows where
the cost lands — most errors are signal events called background and vice-versa near the decision
boundary; choosing the threshold on validation (not the test set) keeps that trade-off honest.

## Which features does it actually rely on?

Notebook 4 left the honest reading of importances for here. Compare LightGBM's built-in **gain**
importance (computed on the training data, MDI-biased — Chapters 06/08/09) with **permutation** importance
measured on **held-out** data: shuffle one feature, see how much PR-AUC drops.

In [ ]:
gain = tuned.booster_.feature_importance(importance_type="gain")
sub = np.random.default_rng(SEED).choice(len(Xte), size=20000, replace=False)
perm = permutation_importance(tuned, Xte.iloc[sub], yte.iloc[sub], n_repeats=5,
                              scoring="average_precision", random_state=SEED, n_jobs=-1)
cols = list(X.columns)
print(f"top-3 by gain (train):        {[cols[i] for i in np.argsort(-gain)[:3]]}")
print(f"top-3 by permutation (test):  {[cols[i] for i in np.argsort(-perm.importances_mean)[:3]]}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8))
viz.plot_feature_importances(cols, gain, top=8, label="gain (train)",
                             color=COLORS["muted"], ax=a1)
viz.plot_feature_importances(cols, perm.importances_mean, std=perm.importances_std, top=8,
                             label="permutation (held-out PR-AUC)", color=COLORS["model"], ax=a2)
plt.show()


**Read the figure.** The two rankings agree on *which handful* of detector features matter, but
**disagree on the order — even at the top**: gain (left) crowns one feature, permutation on held-out data
(right) crowns another. Gain rewards features used in many high-gain training splits; permutation measures
what the model's *held-out* accuracy actually depends on. When you report "the important features," trust
the permutation ranking — it is the one tied to generalization, not to training-set bookkeeping.

## The honest verdict — and when to reach for LightGBM

On MiniBooNE, the three histogram boosters are a **tie on accuracy for any practical purpose** (PR-AUC
within 0.001 once truly matched), and the **speed winner is whatever the matching convention makes it**:
matched capacity → LightGBM; default
shapes → XGBoost; and the default gap closes only past a few million rows. There is **no universal best**.

So when *do* you reach for LightGBM? When its design bites: **very large `n`**, **wide or sparse** feature
spaces (where GOSS's statistical efficiency and EFB's feature bundling pay off), and **native categorical**
data (NB 3's optimal split, no one-hot). On a moderate dense problem like this one, all three are excellent
and interchangeable — so pick on ergonomics, ecosystem, and a quick measurement, not on reputation.

This closes the boosting family (Chapters 06 → 10): bagging (random forests) → AdaBoost → gradient boosting
→ XGBoost → LightGBM. Next, Chapters 11–12 leave trees behind for **neural networks**.

## Your turn

1. **(easy)** Re-run the matched-capacity comparison at `num_leaves = 63` instead of 31. Does the ranking
   (close accuracy, LightGBM fastest) hold, and do all three move together?
2. **(core)** Extend the default-vs-default dial-`n` curve with larger `n` (e.g. 1–2 million) and estimate
   the row count where LightGBM's leaf-wise default would overtake XGBoost's depthwise-6. Why does scale
   help LightGBM here?
3. **(reach)** Bin one continuous feature into a handful of categories (pandas `cut`, dtype `category`) and
   fit LightGBM with native categorical handling (NB 3) versus a one-hot encoding. Compare PR-AUC and fit
   time — when does the native split help?

## What you built

You ran a full, honest workflow on a real, larger dataset and earned the chapter's central claim by
**measuring** it:

- looked at MiniBooNE (imbalance, a 2-D glimpse), set linear and forest **baselines**;
- compared the three histogram boosters at **matched capacity** (tied on PR-AUC within 0.002, LightGBM
  fastest) and **reconciled** that against a default-vs-default flip — the speed gap is **tree shape, not
  the library**;
- dialed `n` up and saw the winner is the **convention**, only slowly the **scale** — **no universal best**;
- confirmed **GOSS**'s efficiency edge appears on real data (matches full, beats uniform 30%);
- shipped a **tuned, early-stopped** LightGBM with a **validation-chosen threshold**, and read **permutation**
  importance for an honest "what matters."

**Vocabulary.** matched-capacity comparison · convention (tree shape) vs scale · PR-AUC / average precision ·
validation-chosen threshold · GOSS efficiency · gain vs permutation importance.

You have finished Chapter 10 — and the boosting family. Next: **Chapter 11, the multi-layer perceptron.**

## References

- Roe, B. P., Yang, H.-J., Zhu, J., et al. (2005). *Boosted decision trees as an alternative to artificial
  neural networks for particle identification.* Nuclear Instruments and Methods in Physics Research A
  543(2–3), 577–584. DOI [10.1016/j.nima.2004.12.018](https://doi.org/10.1016/j.nima.2004.12.018). (The
  MiniBooNE dataset and task.)
- Ke, G., Meng, Q., Finley, T., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision
  Tree.* NeurIPS 30.
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785).
- scikit-learn — `HistGradientBoostingClassifier`, `permutation_importance` (held-out importance).

*Previous: 04 — The estimator and its parameters.*  ·  *End of Chapter 10.*